# M6 — E10: fusion M1 + M2 + multilingual-E5

**Цель.** Проверить, добавляет ли direct dense retrieval от
`multilingual-e5-large-instruct` новые позитивные объявления поверх принятых
M1 lexical и M2 historical sources при строгом лимите в 50 кандидатов.

**Критерий решения.** Основная метрика — macro **Recall@50** на frozen
`benchmark_aligned_proxy_v1`, seed 42. E5 принимается только если хотя бы одна
фиксированная quota policy улучшает M1+M2 control. Все `query_group` в E5
artifact должны точно совпасть с validation split; никаких benchmark labels и
`query_id`-rules не используются.

## План ноутбука

1. Загрузить frozen M0 proxy, materialized M1 texts и E5 validation top-200.
2. Воспроизвести принятый M1 RRF: stemmed full-text BM25 + control title char-TFIDF.
3. Воспроизвести M2 nearest historical query source без validation groups в history.
4. Выделить E5 quota внутри 50 кандидатов и измерить Recall@50 каждой policy.
5. Сохранить таблицу, итоговые validation candidates, manifest и ZIP; залогировать всё в ClearML.

## 1. Конфигурация, пути и live ClearML

In [1]:
from __future__ import annotations

import io
import json
import os
import re
import time
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import snowballstemmer
import sklearn
from clearml import Task
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
TOP_K = 200
FINAL_K = 50
HISTORY_QUOTA = 20
E5_QUOTAS = (0, 5, 10, 15, 20, 25)
RRF_K = 60
BM25_K1, BM25_B = 1.5, 0.75
CHAR_MAX_FEATURES = 200_000
CHAR_BATCH_SIZE = 32

def find_repo_root(start: Path) -> Path:
    """Locate the repository when Jupyter starts the kernel in a subfolder."""
    for candidate in (start, *start.parents):
        if candidate.joinpath(".env").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing .env")


REPO_ROOT = Path(os.environ["AVITO_REPO_ROOT"]) if os.environ.get("AVITO_REPO_ROOT") else find_repo_root(Path.cwd())
DATA_DIR = Path(os.environ.get("AVITO_DATA_DIR", "/Users/kite/Downloads/dataset"))
M1_ARTIFACT_DIR = Path(
    os.environ.get("AVITO_M1_ARTIFACT_DIR", REPO_ROOT / "artifacts/text_preprocessing/m1_lexical_best_v1")
)
E5_ZIP_PATH = Path(
    os.environ.get(
        "AVITO_E5_ZIP_PATH",
        "/Users/kite/Downloads/m3_dense__multilingual_e5_large_instruct.zip",
    )
)
OUTPUT_DIR = REPO_ROOT / "artifacts/fusion/m6_e10_e5_fusion_s42"
TRAIN_PATH = DATA_DIR / "train.parquet"
BENCHMARK_QUERIES_PATH = DATA_DIR / "benchmark_queries.parquet"
BENCHMARK_ITEMS_PATH = DATA_DIR / "benchmark_items.parquet"
NOTEBOOK_STARTED = time.perf_counter()
np.random.seed(SEED)

# query_group is an integer factorization and is therefore version-sensitive.
# These are exactly the package versions recorded in the completed E5 artifact.
if pd.__version__ != "2.3.3" or sklearn.__version__ != "1.6.1":
    raise RuntimeError(
        "M6 must run with pandas==2.3.3 and scikit-learn==1.6.1 to reproduce "
        "the E5 frozen validation groups."
    )


def load_dotenv(path: Path) -> None:
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())


assert REPO_ROOT.joinpath(".env").exists(), "Missing .env with ClearML credentials."
assert all(path.exists() for path in (TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH))
assert M1_ARTIFACT_DIR.joinpath("manifest.json").exists(), "M1 materialized texts are required."
assert E5_ZIP_PATH.exists(), f"Missing E5 ZIP: {E5_ZIP_PATH}"
load_dotenv(REPO_ROOT / ".env")
missing_clearml = [key for key in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY") if not os.environ.get(key)]
if missing_clearml:
    raise RuntimeError(f"Missing ClearML settings: {', '.join(missing_clearml)}")
if os.environ.get("CLEARML_OFFLINE_MODE", "").lower() in {"1", "true", "yes"}:
    raise RuntimeError("M6 requires live ClearML; offline mode is disabled.")

clearml_task = Task.init(
    project_name="avito-retrieval",
    task_name="E10__m1_m2_multilingual_e5_quota_fusion__s42",
    reuse_last_task_id=False,
    auto_connect_arg_parser=False,
    auto_connect_frameworks={"detect_repository": False},
    auto_resource_monitoring=False,
    auto_connect_streams=False,
)
clearml_task.connect(
    {
        "stage": "M6_E10_hybrid_fusion",
        "validation_protocol": "benchmark_aligned_proxy_v1",
        "seed": SEED,
        "top_k": TOP_K,
        "final_k": FINAL_K,
        "history_quota": HISTORY_QUOTA,
        "e5_quota_grid": list(E5_QUOTAS),
        "dense_model": "intfloat/multilingual-e5-large-instruct",
        "m1_source": "E04_rrf_stemmed_full_text_bm25__control_title_char_tfidf",
        "m2_source": "nearest_char_history",
    },
    name="config",
)
clearml_logger = clearml_task.get_logger()
print({"clearml_task_id": clearml_task.id, "seed": SEED, "e5_quotas": E5_QUOTAS})

ClearML Task: created new task id=ef6504e38c2345668cb768eed77cde1a


ClearML results page: https://app.clear.ml/projects/588424e922a44a95aa934ad17ca57931/tasks/ef6504e38c2345668cb768eed77cde1a/output/log


{'clearml_task_id': 'ef6504e38c2345668cb768eed77cde1a', 'seed': 42, 'e5_quotas': (0, 5, 10, 15, 20, 25)}


## 2. Frozen M0 proxy, M1 artifact и E5 candidates

In [2]:
SEARCH_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
TRAIN_COLUMNS = [*SEARCH_COLUMNS, "item_id"]


def canonical_query_frame(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame[SEARCH_COLUMNS].copy()
    for column in ("search_query", "search_infm_params_text"):
        result[column] = (
            result[column].astype("string").fillna("<NA>").str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
        )
    for column in ("search_location_id", "search_is_delivery_search", "search_category"):
        result[column] = result[column].astype("string").fillna("<NA>")
    return result


load_started = time.perf_counter()
train_pairs = pd.read_parquet(TRAIN_PATH, columns=TRAIN_COLUMNS)
benchmark_queries = pd.read_parquet(BENCHMARK_QUERIES_PATH, columns=["query_id", *SEARCH_COLUMNS])
benchmark_item_meta = pd.read_parquet(BENCHMARK_ITEMS_PATH, columns=["item_id", "item_category_id"])
m1_items = pd.read_parquet(
    M1_ARTIFACT_DIR / "benchmark_items_text.parquet",
    columns=["item_id", "item_category_id", "item_text_bm25", "item_title_char_tfidf"],
)
assert np.array_equal(m1_items["item_id"].astype(str).to_numpy(), benchmark_item_meta["item_id"].astype(str).to_numpy())
assert np.array_equal(m1_items["item_category_id"].to_numpy(), benchmark_item_meta["item_category_id"].to_numpy())

train_canonical = canonical_query_frame(train_pairs)
benchmark_canonical = canonical_query_frame(benchmark_queries)
all_contexts = pd.concat([train_canonical, benchmark_canonical], ignore_index=True)
group_ids, _ = pd.factorize(pd.MultiIndex.from_frame(all_contexts), sort=False)
train_pairs["query_group"] = group_ids[: len(train_pairs)]
train_pairs["query_text_norm"] = train_canonical["search_query"].astype(str).to_numpy()
train_pairs["category_key"] = train_canonical["search_category"].astype(str).to_numpy()
benchmark_queries["query_group"] = group_ids[len(train_pairs) :]
benchmark_queries["query_text_norm"] = benchmark_canonical["search_query"].astype(str).to_numpy()
benchmark_queries["category_key"] = benchmark_canonical["search_category"].astype(str).to_numpy()

item_ids = m1_items["item_id"].astype(str).to_numpy()
item_id_set = set(item_ids)
proxy_pairs = train_pairs.loc[train_pairs["item_id"].astype(str).isin(item_id_set)].copy()

with zipfile.ZipFile(E5_ZIP_PATH) as archive:
    members = archive.namelist()
    result_member = next(name for name in members if name.endswith("m3_e05a_results.csv"))
    ranking_member = next(name for name in members if name.endswith("validation_top200__multilingual_e5_large_instruct.jsonl"))
    e5_result = pd.read_csv(io.BytesIO(archive.read(result_member)))
    dense_rows = [json.loads(line) for line in archive.read(ranking_member).decode("utf-8").splitlines()]

dense_by_group = {int(row["query_group"]): list(map(str, row["candidate_item_ids"])) for row in dense_rows}
validation_groups = set(dense_by_group)
assert len(dense_by_group) == len(dense_rows) == 5310
assert validation_groups.issubset(set(proxy_pairs["query_group"].astype(int)))
proxy_valid_pairs = proxy_pairs.loc[proxy_pairs["query_group"].isin(validation_groups)].copy()
validation_queries = (
    proxy_valid_pairs.sort_values("query_group")
    .drop_duplicates("query_group")
    [["query_group", *SEARCH_COLUMNS, "query_text_norm", "category_key"]]
    .reset_index(drop=True)
)
assert len(validation_queries) == len(validation_groups)
assert set(validation_queries["query_group"].astype(int)) == validation_groups
gold_by_group = proxy_valid_pairs.groupby("query_group", sort=False)["item_id"].agg(lambda values: frozenset(values.astype(str))).to_dict()
gold_sets = [gold_by_group[group] for group in validation_queries["query_group"]]
assert all(len(row) == TOP_K and len(row) == len(set(row)) for row in dense_by_group.values())
dense_rankings = [dense_by_group[int(group)] for group in validation_queries["query_group"]]

category_to_indices = {
    str(category): group.index.to_numpy(dtype=np.int64)
    for category, group in m1_items.groupby("item_category_id", sort=False)
}
all_indices = np.arange(len(m1_items), dtype=np.int64)
allowed_indices_by_query = [
    category_to_indices.get(str(category), all_indices)
    for category in validation_queries["search_category"]
]
category_to_item_ids = {
    str(category): set(group["item_id"].astype(str))
    for category, group in m1_items.groupby("item_category_id", sort=False)
}
load_seconds = time.perf_counter() - load_started

print({
    "load_seconds": round(load_seconds, 2),
    "validation_queries": len(validation_queries),
    "validation_positive_rows": len(proxy_valid_pairs),
    "e5_recall@50": round(float(e5_result.iloc[0]["recall@50"]), 6),
    "e5_item_encoding_seconds": round(float(e5_result.iloc[0]["item_encoding_seconds"]), 2),
})

{'load_seconds': 8.03, 'validation_queries': 5310, 'validation_positive_rows': 6633, 'e5_recall@50': 0.293394, 'e5_item_encoding_seconds': 1005.68}


## 3. Воспроизведение принятого M1 lexical source

In [3]:
TOKEN_PATTERN = r"(?u)\b[0-9a-zа-я]{2,}\b"
NON_WORD_RE = re.compile(r"[^0-9a-zа-я]+")
STEMMER = snowballstemmer.stemmer("russian")


def normalize_russian_text(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    return " ".join(NON_WORD_RE.sub(" ", text.lower().replace("ё", "е")).split())


def stem_russian_text(value: object) -> str:
    return " ".join(STEMMER.stemWords(normalize_russian_text(value).split()))


class SparseBM25:
    """Exact M1 BM25, using M1's already materialized stemmed item texts."""

    def __init__(self, k1: float = BM25_K1, b: float = BM25_B, epsilon: float = 0.25):
        self.k1, self.b, self.epsilon = k1, b, epsilon

    def fit(self, documents: list[str]) -> "SparseBM25":
        counts = CountVectorizer(token_pattern=TOKEN_PATTERN, lowercase=False, dtype=np.float32).fit_transform(documents)
        self.doc_len = np.asarray(counts.sum(axis=1)).ravel().astype(np.float32)
        self.matrix = counts.tocsc()
        self.n_docs = counts.shape[0]
        del counts
        df = np.diff(self.matrix.indptr).astype(np.float64)
        idf = np.log((self.n_docs - df + 0.5) / (df + 0.5))
        idf[idf < 0] = self.epsilon * float(idf.mean())
        self.idf = idf.astype(np.float32)
        self.norm = self.k1 * (1 - self.b + self.b * self.doc_len / self.doc_len.mean())
        self.vocabulary = self._vectorizer.vocabulary_ if hasattr(self, "_vectorizer") else None
        return self

    def fit_with_vocabulary(self, documents: list[str]) -> "SparseBM25":
        self._vectorizer = CountVectorizer(token_pattern=TOKEN_PATTERN, lowercase=False, dtype=np.float32)
        counts = self._vectorizer.fit_transform(documents)
        self.doc_len = np.asarray(counts.sum(axis=1)).ravel().astype(np.float32)
        self.matrix = counts.tocsc()
        self.n_docs = counts.shape[0]
        del counts
        df = np.diff(self.matrix.indptr).astype(np.float64)
        idf = np.log((self.n_docs - df + 0.5) / (df + 0.5))
        idf[idf < 0] = self.epsilon * float(idf.mean())
        self.idf = idf.astype(np.float32)
        self.norm = self.k1 * (1 - self.b + self.b * self.doc_len / self.doc_len.mean())
        self.vocabulary = self._vectorizer.vocabulary_
        return self

    def top_k(self, query: str, allowed: np.ndarray, k: int = TOP_K) -> np.ndarray:
        scores = np.zeros(self.n_docs, dtype=np.float32)
        for token, query_tf in Counter(query.split()).items():
            feature = self.vocabulary.get(token)
            if feature is None:
                continue
            start, stop = self.matrix.indptr[feature : feature + 2]
            rows, term_tf = self.matrix.indices[start:stop], self.matrix.data[start:stop]
            scores[rows] += query_tf * self.idf[feature] * term_tf * (self.k1 + 1) / (term_tf + self.norm[rows])
        k = min(k, len(allowed))
        allowed_scores = scores[allowed]
        selected = np.argpartition(allowed_scores, len(allowed) - k)[len(allowed) - k :]
        return allowed[selected[np.argsort(allowed_scores[selected])[::-1]]]


def top_k_from_scores(scores: np.ndarray, k: int) -> np.ndarray:
    selected = np.argpartition(scores, len(scores) - k)[len(scores) - k :]
    return selected[np.argsort(scores[selected])[::-1]]


def rrf_fuse(left: np.ndarray, right: np.ndarray) -> np.ndarray:
    scores: dict[int, float] = {}
    for ranking in (left, right):
        for rank, item_idx in enumerate(ranking, start=1):
            scores[int(item_idx)] = scores.get(int(item_idx), 0.0) + 1.0 / (RRF_K + rank)
    return np.asarray(sorted(scores, key=lambda idx: (-scores[idx], idx))[:TOP_K], dtype=np.int64)


def recall_at(rankings: list[list[str]], k: int) -> float:
    return float(np.mean([len(set(prediction[:k]) & relevant) / len(relevant) for prediction, relevant in zip(rankings, gold_sets, strict=True)]))


def hit_rate_at(rankings: list[list[str]], k: int) -> float:
    return float(np.mean([bool(set(prediction[:k]) & relevant) for prediction, relevant in zip(rankings, gold_sets, strict=True)]))


bm25_queries = [
    # Retained M1 E04 uses only search_query. Filters were tested separately
    # and are deliberately not part of the selected lexical source.
    stem_russian_text(query)
    for query in validation_queries["search_query"]
]
char_queries = [normalize_russian_text(query) for query in validation_queries["search_query"]]

bm25_started = time.perf_counter()
bm25 = SparseBM25().fit_with_vocabulary(m1_items["item_text_bm25"].fillna("").astype(str).tolist())
bm25_rankings = [
    bm25.top_k(query, allowed)
    for query, allowed in zip(bm25_queries, allowed_indices_by_query, strict=True)
]
bm25_seconds = time.perf_counter() - bm25_started
del bm25

char_started = time.perf_counter()
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb", lowercase=False, ngram_range=(3, 5), min_df=2,
    max_features=CHAR_MAX_FEATURES, sublinear_tf=True, dtype=np.float32,
)
char_matrix = char_vectorizer.fit_transform(m1_items["item_title_char_tfidf"].fillna("").astype(str))
query_matrix = char_vectorizer.transform(char_queries)
char_rankings: list[np.ndarray] = []
for start in range(0, query_matrix.shape[0], CHAR_BATCH_SIZE):
    stop = min(start + CHAR_BATCH_SIZE, query_matrix.shape[0])
    scores_batch = (query_matrix[start:stop] @ char_matrix.T).toarray()
    for scores, allowed in zip(scores_batch, allowed_indices_by_query[start:stop], strict=True):
        char_rankings.append(allowed[top_k_from_scores(scores[allowed], TOP_K)])
char_seconds = time.perf_counter() - char_started
del char_matrix, query_matrix, char_vectorizer

m1_rankings = [
    item_ids[rrf_fuse(bm25_ranking, char_ranking)].astype(str).tolist()
    for bm25_ranking, char_ranking in zip(bm25_rankings, char_rankings, strict=True)
]
print({
    "m1_recall@50": round(recall_at(m1_rankings, 50), 6),
    "reference_m1_recall@50": 0.312988,
    "bm25_seconds": round(bm25_seconds, 2),
    "char_seconds": round(char_seconds, 2),
})

{'m1_recall@50': 0.312988, 'reference_m1_recall@50': 0.312988, 'bm25_seconds': 34.92, 'char_seconds': 34.77}


## 4. Воспроизведение M2 nearest historical-query source

In [4]:
history_pairs = train_pairs.loc[
    (~train_pairs["query_group"].isin(validation_groups))
    & train_pairs["item_id"].astype(str).isin(item_id_set)
].copy()
assert history_pairs["query_group"].isin(validation_groups).sum() == 0


def build_history_lookup(frame: pd.DataFrame) -> dict[str, list[str]]:
    unique = frame[["query_group", "query_text_norm", "item_id"]].drop_duplicates()
    counts = unique.groupby(["query_text_norm", "item_id"], sort=False).size().rename("support").reset_index()
    lookup: dict[str, list[str]] = {}
    for query_text, part in counts.groupby("query_text_norm", sort=False):
        part = part.sort_values(["support", "item_id"], ascending=[False, True])
        lookup[str(query_text)] = part["item_id"].astype(str).tolist()
    return lookup


def filter_history_by_category(candidates: list[str], category_key: str) -> list[str]:
    allowed = category_to_item_ids.get(str(category_key), item_id_set)
    return [item_id for item_id in candidates if item_id in allowed][:TOP_K]


history_started = time.perf_counter()
history_lookup = build_history_lookup(history_pairs)
history_texts = np.asarray(sorted(history_lookup), dtype=object)
history_vectorizer = TfidfVectorizer(
    analyzer="char_wb", preprocessor=normalize_russian_text, lowercase=False,
    ngram_range=(3, 5), min_df=1, max_features=CHAR_MAX_FEATURES,
    sublinear_tf=True, dtype=np.float32,
)
history_matrix = history_vectorizer.fit_transform(history_texts)
validation_history_matrix = history_vectorizer.transform(validation_queries["query_text_norm"].tolist())
history_rankings: list[list[str]] = []
for start in range(0, validation_history_matrix.shape[0], CHAR_BATCH_SIZE):
    stop = min(start + CHAR_BATCH_SIZE, validation_history_matrix.shape[0])
    scores_batch = (validation_history_matrix[start:stop] @ history_matrix.T).toarray()
    for scores, category in zip(scores_batch, validation_queries["category_key"].iloc[start:stop], strict=True):
        best_idx = int(np.argmax(scores))
        history_rankings.append(
            filter_history_by_category(history_lookup[str(history_texts[best_idx])], str(category))
            if float(scores[best_idx]) > 0 else []
        )
history_seconds = time.perf_counter() - history_started
del history_matrix, validation_history_matrix, history_vectorizer

print({
    "history_rows": len(history_pairs),
    "history_unique_queries": len(history_texts),
    "history_coverage": round(float(np.mean([bool(row) for row in history_rankings])), 6),
    "history_recall@50": round(recall_at(history_rankings, 50), 6),
    "history_seconds": round(history_seconds, 2),
})

{'history_rows': 26377, 'history_unique_queries': 10368, 'history_coverage': 0.999812, 'history_recall@50': 0.106364, 'history_seconds': 4.43}


## 5. Quota fusion и измерение complementarity

In [5]:
def merge_candidates(
    m1: list[str], history: list[str], dense: list[str], *, history_quota: int, dense_quota: int, priority: str
) -> list[str]:
    selected: list[str] = []

    def append_unique(source: list[str], limit: int | None = None) -> None:
        for item_id in source if limit is None else source[:limit]:
            if item_id not in selected:
                selected.append(item_id)
            if len(selected) == FINAL_K:
                return

    if priority == "history_then_e5":
        append_unique(history, history_quota)
        append_unique(dense, dense_quota)
    elif priority == "e5_then_history":
        append_unique(dense, dense_quota)
        append_unique(history, history_quota)
    else:
        raise ValueError(priority)
    append_unique(m1)
    return selected[:FINAL_K]


def source_overlap(left: list[list[str]], right: list[list[str]]) -> dict[str, float]:
    overlap = [len(set(a[:TOP_K]) & set(b[:TOP_K])) for a, b in zip(left, right, strict=True)]
    return {"mean_top200_overlap": float(np.mean(overlap)), "median_top200_overlap": float(np.median(overlap))}


overlap_table = pd.DataFrame([
    {"pair": "M1__M2", **source_overlap(m1_rankings, history_rankings)},
    {"pair": "M1__E5", **source_overlap(m1_rankings, dense_rankings)},
    {"pair": "M2__E5", **source_overlap(history_rankings, dense_rankings)},
])
display(overlap_table)

fusion_rows: list[dict[str, object]] = []
fusion_predictions: dict[str, list[list[str]]] = {}
for dense_quota in E5_QUOTAS:
    for priority in ("history_then_e5", "e5_then_history"):
        predictions = [
            merge_candidates(m1, history, dense, history_quota=HISTORY_QUOTA, dense_quota=dense_quota, priority=priority)
            for m1, history, dense in zip(m1_rankings, history_rankings, dense_rankings, strict=True)
        ]
        name = f"{priority}__history_{HISTORY_QUOTA}__e5_{dense_quota}"
        fusion_predictions[name] = predictions
        fusion_rows.append({
            "policy": name,
            "priority": priority,
            "history_quota": HISTORY_QUOTA,
            "e5_quota": dense_quota,
            "mean_candidates": float(np.mean([len(row) for row in predictions])),
            "recall@50": recall_at(predictions, 50),
            "hit_rate@50": hit_rate_at(predictions, 50),
        })

fusion_frame = pd.DataFrame(fusion_rows).sort_values(
    ["recall@50", "hit_rate@50", "e5_quota", "priority"],
    ascending=[False, False, True, True],
).reset_index(drop=True)
baseline = fusion_frame.loc[fusion_frame["e5_quota"].eq(0)].iloc[0]
best = fusion_frame.iloc[0]
fusion_frame["gain_vs_no_e5"] = fusion_frame["recall@50"] - float(baseline["recall@50"])
display(fusion_frame)

decision = {
    "baseline_policy": str(baseline["policy"]),
    "baseline_recall@50": float(baseline["recall@50"]),
    "best_policy": str(best["policy"]),
    "best_recall@50": float(best["recall@50"]),
    "e5_gain_vs_no_e5": float(best["recall@50"] - baseline["recall@50"]),
    "accept_e5": bool(best["e5_quota"] > 0 and best["recall@50"] > baseline["recall@50"]),
}
print({key: round(value, 6) if isinstance(value, float) else value for key, value in decision.items()})

,pair,mean_top200_overlap,median_top200_overlap
0,M1__M2,4.556121,2.0
1,M1__E5,82.853484,84.0
2,M2__E5,4.236723,2.0


,policy,priority,history_quota,e5_quota,mean_candidates,recall@50,hit_rate@50,gain_vs_no_e5
0,e5_then_history__history_20__e5_10,e5_then_history,20,10,50.0,0.348471,0.360075,0.003400
1,history_then_e5__history_20__e5_10,history_then_e5,20,10,50.0,0.348471,0.360075,0.003400
2,e5_then_history__history_20__e5_5,e5_then_history,20,5,50.0,0.347484,0.359322,0.002412
3,history_then_e5__history_20__e5_5,history_then_e5,20,5,50.0,0.347484,0.359322,0.002412
4,e5_then_history__history_20__e5_25,e5_then_history,20,25,50.0,0.347355,0.359510,0.002284
5,history_then_e5__history_20__e5_25,history_then_e5,20,25,50.0,0.347355,0.359510,0.002284
6,e5_then_history__history_20__e5_15,e5_then_history,20,15,50.0,0.347325,0.359510,0.002254
7,history_then_e5__history_20__e5_15,history_then_e5,20,15,50.0,0.347325,0.359510,0.002254
8,e5_then_history__history_20__e5_20,e5_then_history,20,20,50.0,0.346283,0.358569,0.001212
9,history_then_e5__history_20__e5_20,history_then_e5,20,20,50.0,0.346283,0.358569,0.001212


{'baseline_policy': 'e5_then_history__history_20__e5_0', 'baseline_recall@50': 0.345071, 'best_policy': 'e5_then_history__history_20__e5_10', 'best_recall@50': 0.348471, 'e5_gain_vs_no_e5': 0.0034, 'accept_e5': True}


## 6. Артефакты, ClearML и ZIP

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results_path = OUTPUT_DIR / "m6_e10_fusion_results.csv"
overlap_path = OUTPUT_DIR / "m6_e10_source_overlap.csv"
manifest_path = OUTPUT_DIR / "m6_e10_manifest.json"
best_candidates_path = OUTPUT_DIR / "validation_final50_candidates.jsonl"

fusion_frame.to_csv(results_path, index=False)
overlap_table.to_csv(overlap_path, index=False)
best_predictions = fusion_predictions[str(best["policy"])]
with best_candidates_path.open("w", encoding="utf-8") as handle:
    for query_group, item_list in zip(validation_queries["query_group"], best_predictions, strict=True):
        assert len(item_list) <= FINAL_K and len(item_list) == len(set(item_list))
        handle.write(json.dumps({"query_group": int(query_group), "candidate_item_ids": item_list}, ensure_ascii=False) + "\n")

manifest = {
    "stage": "M6_E10_m1_m2_e5_quota_fusion",
    "validation_protocol": "benchmark_aligned_proxy_v1",
    "seed": SEED,
    "validation_queries": len(validation_queries),
    "m1_artifact_dir": str(M1_ARTIFACT_DIR),
    "m1_manifest": json.loads((M1_ARTIFACT_DIR / "manifest.json").read_text(encoding="utf-8")),
    "e5_zip_path": str(E5_ZIP_PATH),
    "e5_result": e5_result.iloc[0].to_dict(),
    "config": {"history_quota": HISTORY_QUOTA, "e5_quotas": list(E5_QUOTAS), "final_k": FINAL_K},
    "decision": decision,
    "timing_seconds": {
        "load": load_seconds,
        "m1_bm25": bm25_seconds,
        "m1_char_tfidf": char_seconds,
        "m2_history": history_seconds,
        "notebook_total": time.perf_counter() - NOTEBOOK_STARTED,
    },
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

for _, row in fusion_frame.iterrows():
    clearml_logger.report_scalar("Fusion Recall@50", str(row["policy"]), float(row["recall@50"]), 0)
    clearml_logger.report_scalar("Fusion gain vs no E5", str(row["policy"]), float(row["gain_vs_no_e5"]), 0)
clearml_logger.report_table("M6 E10 quota sweep", "results", 0, table_plot=fusion_frame)
clearml_logger.report_table("M6 E10 source overlap", "top200", 0, table_plot=overlap_table)
clearml_task.set_parameter("results/best_policy", str(best["policy"]))
clearml_task.set_parameter("results/best_recall_at_50", float(best["recall@50"]))
clearml_task.set_parameter("results/accept_e5", bool(decision["accept_e5"]))
clearml_task.upload_artifact("m6_e10_fusion_results", artifact_object=results_path)
clearml_task.upload_artifact("m6_e10_source_overlap", artifact_object=overlap_path)
clearml_task.upload_artifact("m6_e10_manifest", artifact_object=manifest_path)
clearml_task.upload_artifact("m6_validation_final50", artifact_object=best_candidates_path)

from shutil import make_archive
from IPython.display import FileLink, display

zip_path = OUTPUT_DIR.with_suffix(".zip")
make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(OUTPUT_DIR.parent), base_dir=OUTPUT_DIR.name)
assert zip_path.exists()
clearml_task.upload_artifact("m6_e10_artifacts_zip", artifact_object=zip_path)
clearml_task.close()

print({
    "best_policy": decision["best_policy"],
    "best_recall@50": round(decision["best_recall@50"], 6),
    "accept_e5": decision["accept_e5"],
    "output_dir": str(OUTPUT_DIR),
    "zip_path": str(zip_path),
})
display(FileLink(zip_path))

{'best_policy': 'e5_then_history__history_20__e5_10', 'best_recall@50': 0.348471, 'accept_e5': True, 'output_dir': '/Users/kite/Documents/Карьера/avito-services-candidate-generation/artifacts/fusion/m6_e10_e5_fusion_s42', 'zip_path': '/Users/kite/Documents/Карьера/avito-services-candidate-generation/artifacts/fusion/m6_e10_e5_fusion_s42.zip'}


/Users/kite/Documents/Карьера/avito-services-candidate-generation/artifacts/fusion/m6_e10_e5_fusion_s42.zip